In [11]:
import pandas as pd
import requests
import json

import matplotlib.pyplot as plt

In [12]:
prices_df = pd.read_csv('apri_ap_crpouta$defaultview_linear.csv')
print(prices_df.columns)
prices_df.head()


Index(['DATAFLOW', 'LAST UPDATE', 'freq', 'currency', 'prod_veg', 'geo',
       'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG', 'CONF_STATUS'],
      dtype='object')


,DATAFLOW,LAST UPDATE,freq,currency,prod_veg,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:APRI_AP_CRPOUTA$DEFAULTVIEW(1.0),15/05/25 11:00:00,Annual,Euro,Soft wheat - prices per 100 kg,Austria,2015,13.72,NaN,NaN
1,ESTAT:APRI_AP_CRPOUTA$DEFAULTVIEW(1.0),15/05/25 11:00:00,Annual,Euro,Soft wheat - prices per 100 kg,Austria,2016,11.55,NaN,NaN
2,ESTAT:APRI_AP_CRPOUTA$DEFAULTVIEW(1.0),15/05/25 11:00:00,Annual,Euro,Soft wheat - prices per 100 kg,Austria,2017,14.46,NaN,NaN
3,ESTAT:APRI_AP_CRPOUTA$DEFAULTVIEW(1.0),15/05/25 11:00:00,Annual,Euro,Soft wheat - prices per 100 kg,Austria,2018,15.41,NaN,NaN
4,ESTAT:APRI_AP_CRPOUTA$DEFAULTVIEW(1.0),15/05/25 11:00:00,Annual,Euro,Soft wheat - prices per 100 kg,Austria,2019,14.63,NaN,NaN


In [13]:
prices_df = prices_df.drop(['DATAFLOW','LAST UPDATE','freq','OBS_FLAG','CONF_STATUS'],axis=1)
prices_df.head(20)


,currency,prod_veg,geo,TIME_PERIOD,OBS_VALUE
0,Euro,Soft wheat - prices per 100 kg,Austria,2015,13.72
1,Euro,Soft wheat - prices per 100 kg,Austria,2016,11.55
2,Euro,Soft wheat - prices per 100 kg,Austria,2017,14.46
3,Euro,Soft wheat - prices per 100 kg,Austria,2018,15.41
4,Euro,Soft wheat - prices per 100 kg,Austria,2019,14.63
5,Euro,Soft wheat - prices per 100 kg,Austria,2020,14.88
6,Euro,Soft wheat - prices per 100 kg,Austria,2021,22.65
7,Euro,Soft wheat - prices per 100 kg,Austria,2022,28.05
8,Euro,Soft wheat - prices per 100 kg,Austria,2023,17.90
9,Euro,Soft wheat - prices per 100 kg,Austria,2024,17.60


In [14]:
prices_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2001 entries, 0 to 2000
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   currency     2001 non-null   object 
 1   prod_veg     2001 non-null   object 
 2   geo          2001 non-null   object 
 3   TIME_PERIOD  2001 non-null   int64  
 4   OBS_VALUE    1923 non-null   float64
dtypes: float64(1), int64(1), object(3)
memory usage: 78.3+ KB


In [15]:
prices_df.duplicated().sum()
prices_df.isna().sum()

currency        0
prod_veg        0
geo             0
TIME_PERIOD     0
OBS_VALUE      78
dtype: int64

In [16]:
prices_df[prices_df.isnull().any(axis=1)]

,currency,prod_veg,geo,TIME_PERIOD,OBS_VALUE
154,Euro,Soft wheat - prices per 100 kg,Luxembourg,2021,NaN
155,Euro,Soft wheat - prices per 100 kg,Luxembourg,2022,NaN
156,Euro,Soft wheat - prices per 100 kg,Luxembourg,2023,NaN
157,Euro,Soft wheat - prices per 100 kg,Luxembourg,2024,NaN
253,Euro,Durum wheat - prices per 100 kg,Belgium,2015,NaN
...,...,...,...,...,...
1898,National currency,Feed barley - prices per 100 kg,Ireland,2023,NaN
1916,National currency,Feed barley - prices per 100 kg,Luxembourg,2021,NaN
1917,National currency,Feed barley - prices per 100 kg,Luxembourg,2022,NaN
1918,National currency,Feed barley - prices per 100 kg,Luxembourg,2023,NaN


In [17]:
prices_df[prices_df["OBS_VALUE"].isnull()]["geo"].value_counts()

geo
Luxembourg    40
Ireland       22
Slovenia       8
Belgium        4
Czechia        2
Austria        2
Name: count, dtype: int64

In [18]:
len(prices_df[prices_df["geo"] == "Luxembourg"])
len(prices_df[prices_df["geo"] == "Ireland"])
len(prices_df[prices_df["geo"] == "Belgium"])

64

### Note: Half the data for Luxembourg and Ireland are missing; substituted with median for the null values (possible outside influence/biases)

In [19]:
#fill in null values with median
prices_df['OBS_VALUE'] = prices_df.groupby('geo')['OBS_VALUE'].transform(lambda x: x.fillna(x.median()))

prices_df.head()

,currency,prod_veg,geo,TIME_PERIOD,OBS_VALUE
0,Euro,Soft wheat - prices per 100 kg,Austria,2015,13.72
1,Euro,Soft wheat - prices per 100 kg,Austria,2016,11.55
2,Euro,Soft wheat - prices per 100 kg,Austria,2017,14.46
3,Euro,Soft wheat - prices per 100 kg,Austria,2018,15.41
4,Euro,Soft wheat - prices per 100 kg,Austria,2019,14.63


In [20]:
import plotly.express as px

fig = px.line(
    prices_df,
    x="TIME_PERIOD",
    y="OBS_VALUE",
    color="geo",
    markers=True,
    title="crop price by country",
    labels={
        "TIME_PERIOD": "Year",
        "OBS_VALUE": "price",
        "geo": "Country/area",
    },
)

fig.show()

In [21]:
import plotly.express as px

fig = px.line(
    prices_df,
    x="TIME_PERIOD",
    y="OBS_VALUE",
    color="prod_veg",
    markers=True,
    title="crop price by crop",
    labels={
        "TIME_PERIOD": "Year",
        "OBS_VALUE": "price",
        "prod_veg": "crop",
    },
)

fig.show()

In [22]:
belgium = prices_df[prices_df["geo"] == "Belgium"]

fig = px.line(
    belgium,
    x="TIME_PERIOD",
    y="OBS_VALUE",
    color="prod_veg",
    title="Belgium Crop Prices Over Time"
)

fig.show()

In [23]:
# df.head(50)
# df["geo"]["belgium"]

In [24]:
# lat, lon of each capital city
eu_countries = {
    "Austria": (48.21, 16.37),
    "Belgium": (50.85, 4.35),
    "Bulgaria": (42.70, 23.32),
    "Croatia": (45.81, 15.98),
    "Cyprus": (35.17, 33.36),
    "Czechia": (50.08, 14.43),
    "Denmark": (55.68, 12.57),
    "Estonia": (59.44, 24.75),
    "Finland": (60.17, 24.94),
    "France": (48.85, 2.35),
    "Germany": (52.52, 13.41),
    "Greece": (37.98, 23.72),
    "Hungary": (47.50, 19.04),
    "Ireland": (53.35, -6.26),
    "Italy": (41.90, 12.50),
    "Latvia": (56.95, 24.10),
    "Lithuania": (54.68, 25.28),
    "Luxembourg": (49.61, 6.13),
    "Malta": (35.90, 14.51),
    "Netherlands": (52.37, 4.90),
    "Poland": (52.23, 21.01),
    "Portugal": (38.72, -9.14),
    "Romania": (44.43, 26.10),
    "Slovakia": (48.15, 17.11),
    "Slovenia": (46.05, 14.51),
    "Spain": (40.42, -3.70),
    "Sweden": (59.33, 18.07)
}

In [25]:
import requests
import pandas as pd

def get_weather(lat, lon, country, year):
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": f"{year}-01-01",
        "end_date": f"{year}-12-31",
        "daily": "temperature_2m_mean,precipitation_sum",
        "timezone": "auto"
    }

    r = requests.get(url, params=params)
    data = r.json()

    df = pd.DataFrame(data["daily"])

    df["time"] = pd.to_datetime(df["time"])
    df["year"] = df["time"].dt.year

    yearly = df.groupby("year").agg({
        "temperature_2m_mean": "mean",
        "precipitation_sum": "sum"
    }).reset_index()

    yearly["geo"] = country

    return yearly

In [26]:
all_weather = []

for country, (lat, lon) in eu_countries.items():
    for year in range(2015, 2025):
        try:
            df = get_weather(lat, lon, country, year)
            all_weather.append(df)
        except:
            print(f"Failed: {country} {year}")

weather_df = pd.concat(all_weather, ignore_index=True)

In [27]:
print(weather_df)

     year  temperature_2m_mean  precipitation_sum      geo
0    2015            11.941370              561.2  Austria
1    2016            11.251913              704.2  Austria
2    2017            11.234795              527.5  Austria
3    2018            12.166027              614.9  Austria
4    2019            12.056712              614.0  Austria
..    ...                  ...                ...      ...
265  2020             8.789617              612.5   Sweden
266  2021             7.089863              671.4   Sweden
267  2022             7.646027              601.5   Sweden
268  2023             7.100000              772.0   Sweden
269  2024             7.955464              647.9   Sweden

[270 rows x 4 columns]


In [28]:
prices_df["TIME_PERIOD"] = prices_df["TIME_PERIOD"].astype(int)
prices_df = prices_df.rename(columns={"TIME_PERIOD": "year"})
prices_df = prices_df.rename(columns={"OBS_VALUE": "selling price"})

In [29]:
merged_df = prices_df.merge(
    weather_df,
    on=["geo", "year"],
    how="left"
)

In [30]:
print(merged_df)

               currency                         prod_veg       geo  year  \
0                  Euro   Soft wheat - prices per 100 kg   Austria  2015   
1                  Euro   Soft wheat - prices per 100 kg   Austria  2016   
2                  Euro   Soft wheat - prices per 100 kg   Austria  2017   
3                  Euro   Soft wheat - prices per 100 kg   Austria  2018   
4                  Euro   Soft wheat - prices per 100 kg   Austria  2019   
...                 ...                              ...       ...   ...   
1996  National currency  Feed barley - prices per 100 kg  Slovakia  2021   
1997  National currency  Feed barley - prices per 100 kg  Slovakia  2022   
1998  National currency  Feed barley - prices per 100 kg  Slovakia  2023   
1999  National currency  Feed barley - prices per 100 kg  Slovakia  2024   
2000  National currency  Feed barley - prices per 100 kg   Kosovo*  2019   

      selling price  temperature_2m_mean  precipitation_sum  
0             13.72      

In [31]:
merged_df["prod_veg"] = merged_df["prod_veg"].str.replace(r"\s-\sprices.*", "", regex=True)
merged_df.head()

,currency,prod_veg,geo,year,selling price,temperature_2m_mean,precipitation_sum
0,Euro,Soft wheat,Austria,2015,13.72,11.941370,561.2
1,Euro,Soft wheat,Austria,2016,11.55,11.251913,704.2
2,Euro,Soft wheat,Austria,2017,14.46,11.234795,527.5
3,Euro,Soft wheat,Austria,2018,15.41,12.166027,614.9
4,Euro,Soft wheat,Austria,2019,14.63,12.056712,614.0


In [32]:
merged_df

,currency,prod_veg,geo,year,selling price,temperature_2m_mean,precipitation_sum
0,Euro,Soft wheat,Austria,2015,13.72,11.941370,561.2
1,Euro,Soft wheat,Austria,2016,11.55,11.251913,704.2
2,Euro,Soft wheat,Austria,2017,14.46,11.234795,527.5
3,Euro,Soft wheat,Austria,2018,15.41,12.166027,614.9
4,Euro,Soft wheat,Austria,2019,14.63,12.056712,614.0
...,...,...,...,...,...,...,...
1996,National currency,Feed barley,Slovakia,2021,15.51,10.789315,611.2
1997,National currency,Feed barley,Slovakia,2022,24.43,11.939178,527.6
1998,National currency,Feed barley,Slovakia,2023,16.64,12.217260,735.4
1999,National currency,Feed barley,Slovakia,2024,17.89,12.767213,721.5


In [33]:
merged_df.isna().sum()

currency                0
prod_veg                0
geo                     0
year                    0
selling price           0
temperature_2m_mean    23
precipitation_sum      23
dtype: int64

In [34]:
merged_df[merged_df.isnull().any(axis=1)]

,currency,prod_veg,geo,year,selling price,temperature_2m_mean,precipitation_sum
238,Euro,Soft wheat,United Kingdom,2015,17.01,NaN,NaN
239,Euro,Soft wheat,United Kingdom,2016,14.68,NaN,NaN
240,Euro,Soft wheat,United Kingdom,2017,16.62,NaN,NaN
241,Euro,Soft wheat,United Kingdom,2018,18.49,NaN,NaN
242,Euro,Soft wheat,United Kingdom,2019,18.11,NaN,NaN
761,Euro,Barley,United Kingdom,2015,14.41,NaN,NaN
762,Euro,Barley,United Kingdom,2016,12.46,NaN,NaN
763,Euro,Barley,United Kingdom,2017,13.77,NaN,NaN
764,Euro,Barley,United Kingdom,2018,16.75,NaN,NaN
765,Euro,Barley,United Kingdom,2019,14.67,NaN,NaN


In [35]:
merged_df["geo"].unique()

array(['Austria', 'Belgium', 'Bulgaria', 'Cyprus', 'Czechia', 'Germany',
       'Denmark', 'Estonia', 'Greece', 'Spain', 'Finland', 'France',
       'Croatia', 'Hungary', 'Ireland', 'Italy', 'Lithuania',
       'Luxembourg', 'Latvia', 'Netherlands', 'Poland', 'Portugal',
       'Romania', 'Sweden', 'Slovenia', 'Slovakia', 'United Kingdom',
       'Kosovo*'], dtype=object)

In [38]:
merged_df = merged_df[~merged_df["geo"].isin(["United Kingdom", "Kosovo*"])]
merged_df

,currency,prod_veg,geo,year,selling price,temperature_2m_mean,precipitation_sum
0,Euro,Soft wheat,Austria,2015,13.72,11.941370,561.2
1,Euro,Soft wheat,Austria,2016,11.55,11.251913,704.2
2,Euro,Soft wheat,Austria,2017,14.46,11.234795,527.5
3,Euro,Soft wheat,Austria,2018,15.41,12.166027,614.9
4,Euro,Soft wheat,Austria,2019,14.63,12.056712,614.0
...,...,...,...,...,...,...,...
1995,National currency,Feed barley,Slovakia,2020,12.42,11.560383,716.9
1996,National currency,Feed barley,Slovakia,2021,15.51,10.789315,611.2
1997,National currency,Feed barley,Slovakia,2022,24.43,11.939178,527.6
1998,National currency,Feed barley,Slovakia,2023,16.64,12.217260,735.4


In [39]:
merged_df.isna().sum()

currency               0
prod_veg               0
geo                    0
year                   0
selling price          0
temperature_2m_mean    0
precipitation_sum      0
dtype: int64